## Load csv data

In [19]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import os

# data_path = "/home/jude/Documents/humanoid_baseline/robot_lab/outputs/all_data_sim/system_state.csv"
# data = pd.read_csv(data_path)
# print("data length", len(data['value_0']))
# plt.plot(data['timestep'], data['value_0'])
# plt.show()

log_path = "/home/jude/Documents/humanoid_baseline/local/logs/skrl/g1_flat_rwm/exp: noise & entropy/cfdff202a883271a0a108376e4cc0324938bf789_ppo_torch"

system_state_data_path = os.path.join(log_path, "data_recorder/system_state.csv")
system_action_data_path = os.path.join(log_path, "data_recorder/system_action.csv")
system_contact_data_path = os.path.join(log_path, "data_recorder/system_contact.csv")
system_termination_data_path = os.path.join(log_path, "data_recorder/system_termination.csv")

system_state_data = pd.read_csv(system_state_data_path)
system_action_data = pd.read_csv(system_action_data_path)
system_contact_data = pd.read_csv(system_contact_data_path)
system_termination_data = pd.read_csv(system_termination_data_path)

print("system_state_data", system_state_data.shape)
print("system_action_data", system_action_data.shape)
print("system_contact_data", system_contact_data.shape)
print("system_termination_data", system_termination_data.shape)



system_state_data (1000, 56)
system_action_data (1000, 24)
system_contact_data (1000, 3)
system_termination_data (1000, 2)


## Load System Dynamics Model

In [20]:
from skrl.models.rwm_world_model import SystemDynamicsEnsemble
from skrl.resources.preprocessors import EmpiricalNormalization

world_model_path = os.path.join(log_path, "world_model.pt")
world_model, state_normalizer, action_normalizer = SystemDynamicsEnsemble.from_checkpoint(
    world_model_path,
    device="cuda",
)

Model loaded from /home/jude/Documents/humanoid_baseline/local/logs/skrl/g1_flat_rwm/exp: noise & entropy/cfdff202a883271a0a108376e4cc0324938bf789_ppo_torch/world_model.pt
State normalizer loaded from checkpoint
Action normalizer loaded from checkpoint


In [21]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# 准备数据 - 移除timestep列（如果存在）
def prepare_data(df):
    if 'timestep' in df.columns:
        return df.drop('timestep', axis=1).values
    return df.values

# 转换为numpy数组
system_state_array = prepare_data(system_state_data)  # (257, 55) - 移除timestep后
system_action_array = prepare_data(system_action_data)  # (257, 23)

print(f"Original data shapes - State: {system_state_array.shape}, Action: {system_action_array.shape}")
print(f"Original data ranges - State: [{system_state_array.min():.3f}, {system_state_array.max():.3f}]")
print(f"Original data ranges - Action: [{system_action_array.min():.3f}, {system_action_array.max():.3f}]")

Original data shapes - State: (1000, 55), Action: (1000, 23)
Original data ranges - State: [-14.938, 10.018]
Original data ranges - Action: [-7.552, 8.784]


In [22]:
# 应用归一化器到输入数据
def apply_normalization(data, normalizer):
    """应用归一化器到数据"""
    if normalizer is not None:
        # 将数据转换为tensor并应用归一化
        data_tensor = torch.FloatTensor(data).cuda()
        normalized_tensor = normalizer(data_tensor)
        return normalized_tensor.cpu().numpy()
    return data

# 归一化输入数据
normalized_state_array = apply_normalization(system_state_array, state_normalizer)
normalized_action_array = apply_normalization(system_action_array, action_normalizer)

print(f"Normalized data ranges - State: [{normalized_state_array.min():.3f}, {normalized_state_array.max():.3f}]")
print(f"Normalized data ranges - Action: [{normalized_action_array.min():.3f}, {normalized_action_array.max():.3f}]")

Normalized data ranges - State: [-4.208, 3.048]
Normalized data ranges - Action: [-3.978, 3.796]


In [23]:
# 创建滑动窗口（使用归一化数据）
def create_sliding_windows(state_data, action_data, history_horizon=32):
    """
    创建滑动窗口用于世界模型预测
    返回: inputs, targets
    inputs: dict with 'x_state_batch' and 'x_action_batch'
    targets: ground truth values for next timestep (normalized)
    """
    num_sequences = len(state_data) - history_horizon
    
    # 输入序列 (前32个时间步) - 使用归一化数据
    input_states = []
    input_actions = []
    
    # 目标值 (第33个时间步) - 使用归一化数据
    target_states = []
    target_actions = []
    
    for i in range(num_sequences):
        # 输入: 从i到i+history_horizon-1
        input_states.append(state_data[i:i+history_horizon])
        input_actions.append(action_data[i:i+history_horizon])
        
        # 目标: i+history_horizon
        target_states.append(state_data[i+history_horizon])
        target_actions.append(action_data[i+history_horizon])
    
    return {
        'x_state_batch': torch.FloatTensor(np.array(input_states)).cuda(),
        'x_action_batch': torch.FloatTensor(np.array(input_actions)).cuda()
    }, {
        'target_states': np.array(target_states),
        'target_actions': np.array(target_actions)
    }

# 创建窗口（使用归一化数据）
inputs, normalized_targets = create_sliding_windows(normalized_state_array, normalized_action_array, history_horizon=32)

print(f"Created {len(normalized_targets['target_states'])} prediction windows")
print(f"Input shapes - State: {inputs['x_state_batch'].shape}, Action: {inputs['x_action_batch'].shape}")

Created 968 prediction windows
Input shapes - State: torch.Size([968, 32, 55]), Action: torch.Size([968, 32, 23])


In [ ]:
# 世界模型预测（使用归一化输入）
world_model.eval()
with torch.no_grad():
    predictions = world_model.compute(inputs)

# 解析预测结果（仍然是归一化的）
pred_state_means = predictions[0].cpu().numpy()  # 预测状态均值（归一化）
pred_state_stds = predictions[1].cpu().numpy()   # 预测状态标准差（归一化）

print(f"Prediction shapes - State means: {pred_state_means.shape}, stds: {pred_state_stds.shape}")
print(f"Prediction ranges - Means: [{pred_state_means.min():.3f}, {pred_state_means.max():.3f}]")

In [24]:
# 反归一化函数
def denormalize_data(normalized_data, normalizer):
    """反归一化数据"""
    if normalizer is not None:
        # 将归一化数据转换为tensor
        normalized_tensor = torch.FloatTensor(normalized_data).cuda()
        # 反归一化：假设normalizer有inverse方法
        if hasattr(normalizer, 'inverse'):
            denormalized_tensor = normalizer.inverse(normalized_tensor)
            return denormalized_tensor.cpu().numpy()
        else:
            # 如果没有inverse方法，尝试手动反归一化
            # 假设是简单的标准化：x_norm = (x - mean) / std
            if hasattr(normalizer, 'mean') and hasattr(normalizer, 'std'):
                denormalized = normalized_data * normalizer.std.cpu().numpy() + normalizer.mean.cpu().numpy()
                return denormalized
    return normalized_data

# 反归一化预测结果和目标值
denormalized_predictions = denormalize_data(pred_state_means, state_normalizer)
denormalized_targets = denormalize_data(normalized_targets['target_states'], state_normalizer)

print(f"Denormalized prediction ranges: [{denormalized_predictions.min():.3f}, {denormalized_predictions.max():.3f}]")
print(f"Denormalized target ranges: [{denormalized_targets.min():.3f}, {denormalized_targets.max():.3f}]")

Denormalized prediction ranges: [-14.301, 10.595]
Denormalized target ranges: [-14.938, 9.879]


In [25]:
# 计算预测误差（使用反归一化后的数据）
def compute_prediction_errors(predicted, ground_truth):
    """
    计算各种预测误差指标
    """
    mae = np.mean(np.abs(predicted - ground_truth), axis=0)
    mse = np.mean((predicted - ground_truth) ** 2, axis=0)
    rmse = np.sqrt(mse)
    
    print(f"Mean Absolute Error (MAE) per feature: {mae[:5]}...")
    print(f"Root Mean Square Error (RMSE) per feature: {rmse[:5]}...")
    print(f"Overall MAE: {np.mean(mae):.4f}")
    print(f"Overall RMSE: {np.mean(rmse):.4f}")
    
    return {'mae': mae, 'mse': mse, 'rmse': rmse}

# 执行分析
print("\n=== 世界模型预测分析（使用归一化数据）===")
errors = compute_prediction_errors(denormalized_predictions, denormalized_targets)


=== 世界模型预测分析（使用归一化数据）===
Mean Absolute Error (MAE) per feature: [0.04623196 0.04630369 0.05091086 0.20564426 0.20891953]...
Root Mean Square Error (RMSE) per feature: [0.0632818  0.07030702 0.06577098 0.2774031  0.27015248]...
Overall MAE: 0.3279
Overall RMSE: 0.4458


In [ ]:
def plot_all_features(predicted, ground_truth, title, x_min_max, features_per_row=6):
    """
    绘制所有特征的预测对比，动态布局
    
    Args:
        predicted: 预测值 (n_samples, n_features)
        ground_truth: 真实值 (n_samples, n_features)
        title: 图表标题
        features_per_row: 每行显示的特征数量
    """
    n_features = predicted.shape[1]
    n_rows = (n_features + features_per_row - 1) // features_per_row
    
    print(f"Plotting {n_features} features in {n_rows} rows ({features_per_row} per row)")
    
    # 调整图形大小和间距
    fig, axes = plt.subplots(n_rows, features_per_row, figsize=(22, 3.5*n_rows))
    
    # 如果只有一行，确保axes是2D数组
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(n_features):
        row = i // features_per_row
        col = i % features_per_row
        ax = axes[row, col]
        
        # 计算该特征的误差
        error = np.abs(predicted[:, i] - ground_truth[:, i])
        mae = np.mean(error)
        
        # 绘制真实值
        ax.plot(ground_truth[:, i], label='Ground Truth', alpha=0.7, linewidth=1.5)
        
        # 绘制预测值
        ax.plot(predicted[:, i], label='Prediction', alpha=0.7, linewidth=1.5)
        
        # 绘制不确定性区间（需要反归一化std）
        upper_bound = predicted[:, i] + 2 * pred_state_stds * (ground_truth[:, i].max() - ground_truth[:, i].min())
        lower_bound = predicted[:, i] - 2 * pred_state_stds * (ground_truth[:, i].max() - ground_truth[:, i].min())
        # ax.fill_between(range(len(predicted)), lower_bound, upper_bound, 
        #                alpha=0.2, label='±2σ')
        
        # 优化标题和标签样式
        ax.set_title(f'Feature {i+1} (MAE: {mae:.3f})', fontsize=11, fontweight='bold', pad=10)
        ax.set_xlabel('Time Step', fontsize=9)
        ax.set_ylabel('Value', fontsize=9)
        ax.set_xlim(x_min_max[0], x_min_max[1])
        ax.tick_params(labelsize=8)
        ax.grid(True, alpha=0.3, linestyle='--')
        
        # 只在第一行显示legend，优化位置
        if i < features_per_row:
            ax.legend(fontsize=8, loc='upper right', framealpha=0.9)
    
    # 隐藏多余的子图
    for i in range(n_features, n_rows * features_per_row):
        row = i // features_per_row
        col = i % features_per_row
        axes[row, col].set_visible(False)
    
    # 优化整体布局和标题
    plt.suptitle(title, fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout()
    # 为标题留出更多空间
    plt.subplots_adjust(top=0.93, hspace=0.3, wspace=0.25)
    plt.show()

In [ ]:
# 绘制所有特征的预测对比图
plot_all_features(
    denormalized_predictions, 
    denormalized_targets, 
    'World Model: All Features Prediction vs Ground Truth (With Normalization)',
    features_per_row=2,
    x_min_max=(0, 700)
)

In [ ]:
# 绘制误差分布
plt.figure(figsize=(12, 4))

# MAE分布
plt.subplot(1, 2, 1)
plt.bar(range(len(errors['mae'])), errors['mae'])
plt.title('Mean Absolute Error per Feature')
plt.xlabel('Feature Index')
plt.ylabel('MAE')
plt.grid(True, alpha=0.3)

# RMSE分布  
plt.subplot(1, 2, 2)
plt.bar(range(len(errors['rmse'])), errors['rmse'])
plt.title('Root Mean Square Error per Feature')
plt.xlabel('Feature Index')
plt.ylabel('RMSE')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Visualize imagination

In [ ]:
import torch

def system_dynamics_autoregressive_prediction(world_model: SystemDynamicsEnsemble, state_traj, action_traj, extension_traj=None, contact_traj=None, termination_traj=None, eval_trajectory_len=400):
    state_traj_pred = torch.zeros_like(state_traj, device=world_model.device)
    aleatoric_uncertainty_traj_pred = torch.zeros(state_traj.shape[0], state_traj.shape[1], device=world_model.device)
    epistemic_uncertainty_traj_pred = torch.zeros(state_traj.shape[0], state_traj.shape[1], device=world_model.device)
    action_traj_pred = action_traj.clone()
    extension_traj_pred = torch.zeros_like(extension_traj, device=world_model.device) if extension_traj is not None else None
    contact_traj_pred = torch.zeros_like(contact_traj, device=world_model.device) if contact_traj is not None else None
    termination_traj_pred = torch.zeros_like(termination_traj, device=world_model.device) if termination_traj is not None else None
    
    state_traj_pred[:, :world_model.history_horizon] = state_traj[:, :world_model.history_horizon]
    if extension_traj_pred is not None:
        extension_traj_pred[:, :world_model.history_horizon] = extension_traj[:, :world_model.history_horizon]
    if contact_traj_pred is not None:
        contact_traj_pred[:, :world_model.history_horizon] = contact_traj[:, :world_model.history_horizon]
    if termination_traj_pred is not None:
        termination_traj_pred[:, :world_model.history_horizon] = termination_traj[:, :world_model.history_horizon]

    world_model.reset()
    with torch.inference_mode():
        for i in range(world_model.history_horizon, eval_trajectory_len):
            if world_model.architecture_config["type"] in ["rnn", "rssm"] and i > world_model.history_horizon:
                state_input = state_traj_pred[:, i - 1:i]
                action_input = action_traj_pred[:, i - 1:i]
            else:
                state_input = state_traj_pred[:, i - world_model.history_horizon:i]
                action_input = action_traj_pred[:, i - world_model.history_horizon:i]
            _inputs = {"x_state_batch": state_input, "x_action_batch": action_input}
            state_pred, aleatoric_uncertainty, epistemic_uncertainty, extension_pred, contact_pred, termination_pred = world_model.forward(_inputs)
            state_traj_pred[:, i] = state_pred
            aleatoric_uncertainty_traj_pred[:, i] = aleatoric_uncertainty
            epistemic_uncertainty_traj_pred[:, i] = epistemic_uncertainty
            if extension_traj_pred is not None and extension_pred is not None:
                extension_traj_pred[:, i] = extension_pred
            if contact_traj_pred is not None and contact_pred is not None:
                contact_traj_pred[:, i] = torch.sigmoid(contact_pred).round().int()
            if termination_traj_pred is not None and termination_pred is not None:
                termination_traj_pred[:, i] = torch.sigmoid(termination_pred).round().int()
    return state_traj_pred, aleatoric_uncertainty_traj_pred, epistemic_uncertainty_traj_pred, action_traj_pred, extension_traj_pred, contact_traj_pred, termination_traj_pred

In [ ]:
normalized_state_tensor = torch.tensor(normalized_state_array, dtype=torch.float32, device=world_model.device).unsqueeze(0)
normalized_action_tensor = torch.tensor(normalized_action_array, dtype=torch.float32, device=world_model.device).unsqueeze(0)

print(f"World model state dim: {world_model.state_dim}")
print(f"World model action dim: {world_model.action_dim}")
print(f"Input state shape: {normalized_state_tensor.shape}")
print(f"Input action shape: {normalized_action_tensor.shape}")

_return_tuples = system_dynamics_autoregressive_prediction(world_model, normalized_state_tensor, normalized_action_tensor)
normalized_state_tensor_pred = _return_tuples[0]
print(f"Predicted state shape: {normalized_state_tensor_pred.shape}")


In [ ]:
# 提取第一个轨迹的预测结果和真实值进行对比
# normalized_state_tensor_pred 的形状应该是 [1, 400, 55]
# normalized_state_tensor 的形状应该是 [1, 1000, 55]

normalized_state_plot = normalized_state_tensor.cpu().numpy()
normalized_state_pred_plot = normalized_state_tensor_pred.cpu().numpy()


# 只取前400个时间步进行对比
true_trajectory = normalized_state_plot[0, :400]  # [400, 55]
pred_trajectory = normalized_state_pred_plot[0, :400]  # [400, 55]

# 计算每个特征的MAE
mae_per_feature = np.mean(np.abs(true_trajectory - pred_trajectory), axis=0)

# 创建对比图
def plot_prediction_comparison(true_data, pred_data, mae_per_feature, features_per_row=6):
    """
    绘制预测值与真实值的对比图
    
    Args:
        true_data: 真实值 [time, features]
        pred_data: 预测值 [time, features]
        mae_per_feature: 每个特征的MAE
        features_per_row: 每行显示的特征数
    """
    n_features = true_data.shape[1]
    n_rows = (n_features + features_per_row - 1) // features_per_row
    
    fig, axes = plt.subplots(n_rows, features_per_row, figsize=(22, 3.5*n_rows))
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(n_features):
        row = i // features_per_row
        col = i % features_per_row
        ax = axes[row, col]
        
        # 绘制真实值和预测值
        ax.plot(true_data[:, i], label='Ground Truth', alpha=0.8, linewidth=1.5, color='blue')
        ax.plot(pred_data[:, i], label='Prediction', alpha=0.8, linewidth=1.5, color='red', linestyle='--')
        
        # 设置标题和标签
        ax.set_title(f'Feature {i+1} (MAE: {mae_per_feature[i]:.3f})', 
                    fontsize=10, fontweight='bold')
        ax.set_xlabel('Time Step', fontsize=9)
        ax.set_ylabel('Value', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=8)
        
        # 只在第一行显示legend
        if i < features_per_row:
            ax.legend(fontsize=8, loc='upper right')
    
    # 隐藏多余的子图
    for i in range(n_features, n_rows * features_per_row):
        row = i // features_per_row
        col = i % features_per_row
        axes[row, col].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # 打印总体统计
    print(f"Overall MAE: {np.mean(mae_per_feature):.4f}")
    print(f"Best feature: {np.argmin(mae_per_feature)+1} (MAE: {np.min(mae_per_feature):.4f})")
    print(f"Worst feature: {np.argmax(mae_per_feature)+1} (MAE: {np.max(mae_per_feature):.4f})")

# 调用绘图函数
plot_prediction_comparison(true_trajectory, 
                          pred_trajectory, 
                          mae_per_feature)

In [30]:
torch.set_printoptions(sci_mode=False)
state_normalizer.std

tensor([0.5776, 0.5546, 0.2973, 0.9433, 1.0355, 2.3903, 0.1302, 0.1038, 0.0794,
        0.1285, 0.1170, 0.1278, 0.1984, 0.1775, 0.1155, 0.1334, 0.1284, 0.1273,
        0.2106, 0.1755, 0.1169, 0.1273, 0.0808, 0.0728, 0.0931, 0.0909, 0.0929,
        0.0887, 0.0789, 0.0952, 0.0939, 0.0851, 2.4127, 2.2444, 3.2460, 3.9498,
        3.2719, 3.5705, 2.4824, 2.2158, 3.3683, 4.3012, 3.4199, 3.5590, 3.8356,
        1.6967, 1.4250, 1.9927, 2.0263, 2.0796, 1.6271, 1.4369, 2.0267, 1.9507,
        2.0498], device='cuda:0')

In [31]:
state_normalizer.mean

tensor([     0.0278,     -0.0103,     -0.0203,      0.0041,      0.0207,
             0.0305,     -0.0635,      0.0159,     -0.9806,     -0.0073,
             0.0951,     -0.0815,      0.2277,     -0.1232,     -0.0904,
            -0.0192,     -0.0665,      0.0784,      0.2002,     -0.0779,
             0.0957,      0.0105,     -0.0081,     -0.0100,     -0.0026,
            -0.0030,     -0.0051,     -0.0202,      0.0477,      0.0117,
             0.0158,      0.0307,      0.0445,      0.0084,      0.0212,
             0.0197,     -0.0592,      0.0886,      0.0425,     -0.0120,
            -0.0152,      0.0052,     -0.0616,     -0.0431,      0.0003,
             0.0009,     -0.0090,      0.0003,     -0.0010,      0.0006,
            -0.0023,      0.0109,     -0.0010,     -0.0002,      0.0022],
       device='cuda:0')

In [32]:
action_normalizer.std

tensor([0.6609, 0.6981, 0.7380, 1.0410, 2.2537, 1.7389, 0.6790, 0.7146, 0.7508,
        1.1105, 2.2234, 1.7511, 0.6736, 1.5468, 1.4923, 1.6829, 1.6950, 1.7171,
        1.5086, 1.5127, 1.6980, 1.6702, 1.6915], device='cuda:0')

In [33]:
action_normalizer.mean

tensor([-0.0836,  0.3482, -0.2920,  0.5556,  0.2382, -0.4207, -0.1070, -0.2442,
         0.2928,  0.4680,  0.3441,  0.3856,  0.0408, -0.0150, -0.0141, -0.0120,
        -0.1280, -0.0219, -0.0756,  0.1756,  0.0489, -0.0524,  0.1277],
       device='cuda:0')